In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))
from scraping.d2l_scraper import crawler, create_session


In [ ]:
# session=create_session()
# documents = crawler("https://d2l.ai/", session, initial_referer="https://www.google.com/")

In [ ]:
# documents

In [ ]:
from database.vector_storage import save_vault, load_vault

from chunking.fixed_chunker import fixed_chunk_documents
from chunking.recursive_chunker import recursive_chunk_documents
from chunking.semantic_chunker import semantic_chunk_documents
from chunking.code_aware_chunker import code_aware_chunk_documents
from chunking.header_chunker import header_aware_chunk_documents 
from chunking.sliding_chunker import sliding_chunk_documents
from embeddings.embedder import embed_documents,load_embedding_model,embed_query

model = load_embedding_model()


[Init] Waking up RTX 5060 & loading BAAI/bge-base-en-v1.5...
[Init] GPU Model locked to CUDAExecutionProvider in 1.27s


In [ ]:


VAULT_DIR = r"C:\AI_PROJECTS\ice_pytorch\monarch-rag\Monarch-RAG\data\vault_DeepDiveInDeepLearning"

# 1. Try to load from disk first
my_vault = load_vault(VAULT_DIR)

# 2. If it doesn't exist, calculate it on the GPU, then save it
if my_vault is None:
    print("No local vault found. Generating embeddings on GPU...")
    
    # fixed_chunks = fixed_chunk_documents(documents, chunk_size=512)
    header_chunks = header_aware_chunk_documents(documents)
    sliding_chunks = sliding_chunk_documents(documents, chunk_size=512, overlap=128)
    recursive_chunks = recursive_chunk_documents(documents, chunk_size=512, overlap=128)
    code_chunks = code_aware_chunk_documents(documents, max_chars=512, overlap=128)
    semantic_chunks = semantic_chunk_documents(documents, model)

    header_embeddings = embed_documents(header_chunks,model)
    sliding_embeddings = embed_documents(sliding_chunks,model)
    recursive_embeddings = embed_documents(recursive_chunks,model)
    code_embeddings = embed_documents(code_chunks,model)
    semantic_embeddings = embed_documents(semantic_chunks,model)
    # fixed_embeddings = embed_documents(fixed_chunks,model)
    
    my_vault = {
        # "Fixed Size": {"chunks": fixed_chunks, "matrix": fixed_embeddings},
        "Sliding window": {"chunks": sliding_chunks, "matrix": sliding_embeddings},
        "Recursive": {"chunks": recursive_chunks, "matrix": recursive_embeddings},
        "Header Aware": {"chunks": header_chunks, "matrix": header_embeddings},
        "Code Aware Sliding": {"chunks": code_chunks, "matrix": code_embeddings},
        "Semantic": {"chunks": semantic_chunks, "matrix": semantic_embeddings}
    }
    
    # Save it to disk permanently
    save_vault(my_vault, VAULT_DIR)

# 3. Proceed directly to RRF Fusion and Battle Arena!
# run_rrf_fusion("How to check device?", model, my_vault)

# With this in place, your workflow becomes completely decoupled:
# 1. **Kernel Restart:** Takes 0 seconds.
# 2. **Reloading Embeddings:** Takes `~0.3` seconds (NumPy binary loads are blazingly fast).
# 3. **Experimenting with RRF:** You can edit your Reciprocal Rank Fusion formulas, test new queries, and analyze the outputs instantly without ever waking up the RTX 5060 to re-embed the text.

Vault loaded from disk! Restored 5 indexing strategies.


In [ ]:
query = "Expected all tensors to be on the same device, but found at least two devices"
query_embedding = embed_query(query, model)


In [ ]:
# from retrieval.similarity import get_top_k_similar_documents
# top_k_header = get_top_k_similar_documents(query_embedding, header_embeddings, header_chunks, k=3)
# top_k_sliding = get_top_k_similar_documents(query_embedding, sliding_embeddings, sliding_chunks, k=3)
# top_k_recursive = get_top_k_similar_documents(query_embedding, recursive_embeddings, recursive_chunks, k=3)
# top_k_semantic = get_top_k_similar_documents(query_embedding, semantic_embeddings, semantic_chunks, k=3)

In [ ]:

from evaluation.retrieval_evaluator import benchmark_chunking_strategies,execute_rrf_fusion,run_stress_test_suite

benchmark_chunking_strategies(
    "How do I put a tensor on the GPU?", model, my_vault, top_k=2
)


🔍 QUERY: 「 How do I put a tensor on the GPU? 」

📦 STRATEGY: 【 Code Aware Sliding 】
  #1 [Sim: 0.7941] ── [use-gpu > By default JAX puts arrays to GPUs or TPUs if available > 6.7.3. Neural Networks and GPUs ¶]
      "We will see many more examples of how to run models on GPUs in the following chapters, simply because the models will become somewhat more computationally intensive. ⏎  ⏎ For examp..."
  #2 [Sim: 0.7907] ── [use-gpu > By default JAX puts arrays to GPUs or TPUs if available]
      "``` ⏎ <tf.Tensor: shape=(2, 3), dtype=float32, numpy= ⏎ array([[1., 1., 1.], ⏎        [1., 1., 1.]], dtype=float32)> ⏎ ``` ⏎  ⏎ Assuming that you have at least two GPUs, the follow..."

📦 STRATEGY: 【 Header Aware 】
  #1 [Sim: 0.7980] ── [use-gpu > 6.7. GPUs ¶ Colab [pytorch] Open the notebook in Colab Colab [mxnet] Open the notebook in Colab Colab [jax] Open the notebook in Colab Colab [tensorflow] Open the notebook in Colab SageMaker Studio Lab Open the notebook in SageMaker Studio Lab > 6.7.2. 

'Sliding Window'

In [ ]:
test_queries = [
    # Group 1: Syntax
    "What is the exact code to manually set the random seed for CPU and CUDA?",
    "Show me the code to check which device a model's parameters are sitting on.",
    # Group 2: Semantic Gaslight
    "Why am I getting the RuntimeError: Expected all tensors to be on the same device?",
    "How do I fix a shape mismatch error inside nn.Linear forward pass?",
    # Group 3: Needle in the Haystack
    "Does the Zero to Mastery PyTorch course cover PyTorch version 2.0?",
    "What specific data science bootcamp is recommended as a prerequisite before taking this course?",
    # Group 4: Conceptual
    "What is the overarching computer vision project built throughout the milestone chapters called?",
    "What is the difference between torch.rand and torch.randn?",
]
d2l_test_queries = [
    # Group 1: Syntax & Implementation (Tests Code-Aware/AST Chunker)
    "Show me the code to implement scaled dot-product attention from scratch.",
    "How do I initialize the weights and biases for a Multi-Layer Perceptron (MLP) in PyTorch?",
    "What is the mathematical formula for the softmax function as defined in the linear classification chapter?",
    "Show me the exact implementation of the `d2l.Timer` class used for benchmarking.",
    "What is the code snippet to apply gradient clipping in an RNN?",

    # Group 2: Semantic Gaslight & Debugging (Tests Semantic Distance & Context)
    "Why does my deep neural network suffer from vanishing gradients, and how do ResNets fix this?",
    "Why is my validation loss diverging while training loss approaches zero, and what regularization techniques are suggested?",
    "How exactly does dropout behave differently during the training forward pass compared to the testing/inference phase?",
    "What happens if I forget to zero the gradients before calling `backward()` in an optimization loop?",
    "Why do we use convolutions instead of fully connected dense layers for image processing?",

    # Group 3: Needle in the Haystack (Tests Absolute Character Tracking & Specific Lookups)
    "What specific dataset is used as the primary benchmark for the initial image classification models in Chapter 3?",
    "How many layers are in the original AlexNet architecture described in the CNN chapter?",
    "What are the three specific properties of attention mechanisms mentioned in the introductory attention chapter?",
    "In the historical overview, who is credited with popularizing the backpropagation algorithm?",
    "What are the exact dimensions of the images in the Fashion-MNIST dataset?",

    # Group 4: Conceptual & Theoretical (Tests the Parent Reader's ability to provide full narrative context)
    "What is the fundamental difference between cross-entropy loss and mean squared error (MSE)?",
    "How does a Gated Recurrent Unit (GRU) differ structurally from a Long Short-Term Memory (LSTM) cell?",
    "Explain the concept of weight decay and how it mathematically relates to L2 regularization.",
    "What is the purpose of positional encoding in the Transformer architecture, and what functions are used to compute it?",
    "How does Batch Normalization work during training versus inference, and why is it used?",
]
for i in range(len(d2l_test_queries)):
    execute_rrf_fusion(d2l_test_queries[i], model,my_vault, top_k=3)


🧬 FUSED RETRIEVAL FOR: 「 Show me the code to implement scaled dot-product attention from scratch. 」
 #1 [RRF Score: 0.15154] ── attention-scoring-functions::11.3. Attention Scoring Functions ¶ Colab [pytorch] Open the notebook in Colab Colab [mxnet] Open the notebook in Colab Colab [jax] Open the notebook in Colab Colab [tensorflow] Open the notebook in Colab SageMaker Studio Lab Open the notebook in SageMaker Studio Lab > 11.3.3. Scaled Dot Product Attention ¶
     "Let’s return to the dot product attention introduced in (11.3.2) . In general, it requires that both the query and the key have the same vector length, say \(d\..."

 #2 [RRF Score: 0.04353] ── attention-scoring-functions::11.3. Attention Scoring Functions ¶ Colab [pytorch] Open the notebook in Colab Colab [mxnet] Open the notebook in Colab Colab [jax] Open the notebook in Colab Colab [tensorflow] Open the notebook in Colab SageMaker Studio Lab Open the notebook in SageMaker Studio Lab > 11.3.6. Exercises ¶
     "* Implem

In [ ]:
run_stress_test_suite(model,my_vault,d2l_test_queries)


🚀 INITIATING BLIND STRESS TEST SUITE...

🔍 QUERY: 「 Show me the code to implement scaled dot-product attention from scratch. 」

📦 STRATEGY: 【 Code Aware Sliding 】
  #1 [Sim: 0.7427] ── [attention-scoring-functions > 11.3. Attention Scoring Functions ¶ Colab [pytorch] Open the notebook in Colab Colab [mxnet] Open the notebook in Colab Colab [jax] Open the notebook in Colab Colab [tensorflow] Open the notebook in Colab SageMaker Studio Lab Open the notebook in SageMaker Studio Lab > 11.3.5. Summary ¶]
      "## 11.3.6. Exercises ¶ ⏎  ⏎ * Implement distance-based attention by modifying the DotProductAttention code. Note that you only need the squared norms of the keys \(\|\mathbf{k}_i\|..."

📦 STRATEGY: 【 Header Aware 】
  #1 [Sim: 0.7393] ── [attention-scoring-functions > 11.3. Attention Scoring Functions ¶ Colab [pytorch] Open the notebook in Colab Colab [mxnet] Open the notebook in Colab Colab [jax] Open the notebook in Colab Colab [tensorflow] Open the notebook in Colab SageMaker Stud

In [ ]:
from pipeline.indexer import link_prechunked_parents_and_children, load_json_chunks
from retrieval.retrieverPC import build_parent_child_vault
from database.vault_PC_d2l import assemble_d2l_vault
from pathlib import Path

print("Loading JSON chunks from disk...")

# 1. Load the Parent candidates
header_aware_chunks = load_json_chunks(r"C:\AI_PROJECTS\ice_pytorch\monarch-rag\Monarch-RAG\data\vault_DeepDiveInDeepLearning\header_aware\chunks.json")

# 2. Load the Child candidates
code_aware_chunks = load_json_chunks(r"C:\AI_PROJECTS\ice_pytorch\monarch-rag\Monarch-RAG\data\vault_DeepDiveInDeepLearning\code_aware_sliding\chunks.json")
semantic_chunks = load_json_chunks(r"C:\AI_PROJECTS\ice_pytorch\monarch-rag\Monarch-RAG\data\vault_DeepDiveInDeepLearning\semantic\chunks.json")

# 3. ⭐️ Combine the children into a single massive search pool!
all_children_combined = code_aware_chunks + semantic_chunks

# 4. Link them together
split_data = link_prechunked_parents_and_children(
    raw_parents=header_aware_chunks, 
    raw_children=all_children_combined   # <--- Passing the combined list here
)

# --- REPLACE STEP 5 AND 6 IN YOUR NOTEBOOK WITH THIS ---

from retrieval.retrieverPC import build_parent_child_vault
from database.vault_storage_PC import save_vault

# 1. Use the data you already linked in memory
parents = split_data["parents"]
children = split_data["children"]
output_path = r"C:\AI_PROJECTS\ice_pytorch\monarch-rag\Monarch-RAG\data\vault_DeepDiveInDeepLearning"

# 2. Build the Vault directly using your model
print("🚀 Embedding combined children on RTX 5060 (8GB VRAM)...")
d2l_vault = build_parent_child_vault(parents, children, model)

# 3. Save it directly using your vector_storage.py
save_vault(d2l_vault, output_path)

print("🎉 Multi-Child D2L Vault is compiled and saved!")

print("hi bro Multi-Child D2L Vault is compiled and saved!")

Loading JSON chunks from disk...
🔗 Linking 2187 Parents and 9775 Children via Universal Anchors...
✅ Successfully linked 2187 Parents and 9775 Children.
🚀 Embedding combined children on RTX 5060 (8GB VRAM)...
Building Vault: 2187 Parents, 9775 Children...
Vault saved to C:\AI_PROJECTS\ice_pytorch\monarch-rag\Monarch-RAG\data\vault_DeepDiveInDeepLearning*
🎉 Multi-Child D2L Vault is compiled and saved!
hi bro Multi-Child D2L Vault is compiled and saved!


In [ ]:
# from retrieval.retrieverPC import build_parent_child_vault
# from database.vector_storage import save_vault

# # 1. Use the data you already linked in memory
# parents = split_data["parents"]
# children = split_data["children"]
# output_path = r"C:\AI_PROJECTS\ice_pytorch\monarch-rag\Monarch-RAG\data\vault_DeepDiveInDeepLearning"

# # 2. Build the Vault (Directly calling the retriever logic)
# print("🚀 Embedding children on RTX 5060...")
# d2l_vault = build_parent_child_vault(parents, children, model)

# # 3. Save it securely
# save_vault(d2l_vault, output_path)

# print(f"🎉 Vault successfully assembled and saved to {output_path}")

In [ ]:
# from retrieval.retrieverPC import build_parent_child_vault,retrieve_context_for_agent
# vault=build_parent_child_vault(parent_chunks=my_vault["Header Aware"]["chunks"], child_chunks=(my_vault["Code Aware Sliding"]["chunks"]+my_vault["Semantic"]["chunks"]), model=model)
# vault

In [ ]:
# retrieve_context_for_agent("How do I put a tensor on the GPU?",model, vault,3)

In [ ]:
# from database.vault_storage_PC import save_vault,load_vault

# VAULT_DIR = r"C:\AI_PROJECTS\ice_pytorch\monarch-rag\Monarch-RAG\data\vault_DeepDiveInDeepLearning"
# save_vault(vault, VAULT_DIR)

Vault saved to C:\AI_PROJECTS\ice_pytorch\monarch-rag\Monarch-RAG\data\local_vector_vault*
